# 👷 work on 16.05 so far includes:
- benchmakring the uniforms
- one hybrid uniform (laura + me)

In [14]:
import igraph as ig
import numpy as np
import timeit
import statistics

import infomap_funcs as laura
from map_equation import compute_description_length_uniform as mine

In [15]:
def generate_sbm(n, c, p_in, p_out, directed=False, weighted=False):
    """
    Generate a Stochastic Block Model graph.

    Parameters
    ----------
    n        : total number of nodes
    c        : number of communities
    p_in     : within-community connection probability
    p_out    : between-community connection probability
    directed : whether the graph should be directed
    weighted : whether the graph should be weighted 
               (if True, assigns uniform weights from 0 to 1 to edges)

    Returns
    -------
    g : igraph.Graph
    """

    # determine module sizes (distributing nodes as evenly as possible)
    module_sizes = np.full(c, n // c, dtype=int)
    module_sizes[:n % c] += 1   # distribute remainder evenly
    
    # build preference matrix for SBM
    pref_matrix = np.full((c, c), p_out)
    np.fill_diagonal(pref_matrix, p_in)

    # generate graph via igraph's SBM generator
    g = ig.Graph.SBM(
            pref_matrix.tolist(),
            module_sizes.tolist(),
            directed=directed,
            allowed_edge_types='simple'
        )  
    
    # explicitely assign community membership as vertex attribute
    g.vs["community"] = np.repeat(np.arange(c), module_sizes)

    # if weighted, assign random weights to edges
    if weighted:
        g.es["weight"] = np.random.rand(g.ecount())

    return g

In [25]:
n_runs = 5
n_trials = 20

configs = [
    {"n": 100,  "c": 4, "p_in": 0.2,  "p_out": 0.01},
    {"n": 500,  "c": 4, "p_in": 0.1,  "p_out": 0.005},
    {"n": 1000, "c": 4, "p_in": 0.05, "p_out": 0.001},
    #{"n": 2000, "c": 4, "p_in": 0.03, "p_out": 0.0005},
]

for cfg in configs:
    print(f"\n{'#'*35}")
    print(f" n={cfg['n']}, c={cfg['c']}, p_in={cfg['p_in']}, p_out={cfg['p_out']}")
    print(f"{'#'*35}")
    
    # build graph for each of the 4 directed × weighted combinations
    variants = [
        ("undirected_unweighted", False, False),
        ("undirected_weighted",   False, True),
        ("directed_unweighted",   True,  False),
        ("directed_weighted",     True,  True),
    ]
    
    for label, directed, weighted in variants:
        g = generate_sbm(n=cfg["n"], c=cfg["c"], p_in=cfg["p_in"], p_out=cfg["p_out"],
                          directed=directed, weighted=weighted)
        c_partition = g.community_infomap(edge_weights="weight" if weighted else None)
        partition = c_partition.membership

        L_mine = mine(g, partition)
        L_laura = laura.compute_description_length(g, partition)

        mine_times = [timeit.timeit(lambda: mine(g, partition), number=n_trials) for _ in range(n_runs)]
        laura_times = [timeit.timeit(lambda: laura.compute_description_length(g, partition), number=n_trials) for _ in range(n_runs)]

        mine_mean = statistics.mean(mine_times) * 1000 / n_trials
        mine_std  = statistics.stdev(mine_times) * 1000 / n_trials
        laura_mean = statistics.mean(laura_times) * 1000 / n_trials
        laura_std  = statistics.stdev(laura_times) * 1000 / n_trials

        print(f"\n  {label:25s}")
        print(f"    mine  = {mine_mean:7.2f} ± {mine_std:5.2f} ms")
        print(f"    laura = {laura_mean:7.2f} ± {laura_std:5.2f} ms")


###################################
 n=100, c=4, p_in=0.2, p_out=0.01
###################################

  undirected_unweighted    
    mine  =    0.64 ±  0.13 ms
    laura =    0.55 ±  0.02 ms

  undirected_weighted      
    mine  =    0.71 ±  0.12 ms
    laura =    0.79 ±  0.11 ms

  directed_unweighted      
    mine  =    9.10 ±  1.16 ms
    laura =    7.55 ±  1.05 ms

  directed_weighted        
    mine  =   10.31 ±  0.80 ms
    laura =    9.79 ±  0.12 ms

###################################
 n=500, c=4, p_in=0.1, p_out=0.005
###################################

  undirected_unweighted    
    mine  =    3.87 ±  0.32 ms
    laura =    3.11 ±  0.11 ms

  undirected_weighted      
    mine  =    4.25 ±  0.35 ms
    laura =    4.49 ±  0.17 ms

  directed_unweighted      
    mine  =   77.61 ±  0.78 ms
    laura =   78.90 ±  2.53 ms

  directed_weighted        
    mine  =   95.33 ±  0.97 ms
    laura =  110.89 ±  6.47 ms

###################################
 n=1000, c=4, p_in=0

In [ ]:
def compute_description_length_uniform(g, communities, tau=0.15):
    """Hybrid uniform-teleportation implementation of the map equation.
    
    Combines the faster code paths from two independent implementations
    (Savina + Laura). Both produce same L values on every test + keeping the fastest one:
    
    - For undirected graphs => Laura's
        my version built weight arrays using a python list comprehension [1.0] * g.ecount() 
        and then converted that to a numpy array. Laura built the array directly 
        with np.ones(g.ecount(), dtype=np.float64) skipping an entire list creation step.
    
    - For directed graphs => mine
        Laura's pagerank update wrote the iteration as three separate numpy expressions added together,
        so three separate numpy operations, instead of folding them together so fewer numpy calls.
        
    Args:
        g (ig.Graph): Input graph (directed or undirected, weighted or unweighted).
        communities (list[int]): Community label for each node.
        tau (float, optional): Teleportation probability for directed graphs.
            Defaults to 0.15 (PageRank standard damping factor 0.85).
    
    Returns:
        float: Description length L (in bits) of the given partition.
    """
    communities = np.array(communities)
    num_comms = max(communities) + 1
    N = g.vcount()

    if g.is_directed():
        # DIRECTED
        # The walker either follows an outgoing edge or teleports to any
        # node uniformly. Teleportation steps are "recorded" so they
        # contribute to the per-community exit probability via the # τ·(1 - n_α/N)·p_α term.
        
        # Get the (possibly weighted) adjacency matrix
        adj = np.array(g.get_adjacency(
            attribute="weight" if g.is_weighted() else None).data,
            dtype=float
        )
        
        # Compute stationary node visit rates via pagerank
        p = pagerank_uniform(adj, tau=tau)

        # Aggregate node visit rates to per-community visit rates
        p_mod = np.zeros(num_comms)
        np.add.at(p_mod, communities, p)

        # Compute the rate at which the walker leaves each community
        # via outgoing edges (no teleportation contribution yet)
        exit_flow = compute_exit_flow_nonuniform(g, communities, p)

        # Build q_mod, the total exit probability per community,
        # combining exit flow from edges + the teleportation term
        # The factor (1 - n_α/N) is the probability that a uniform
        # teleportation lands outside community α
        n_mod = np.bincount(communities, minlength=num_comms).astype(float)
        q_mod = tau * (1 - n_mod / N) * p_mod + (1 - tau) * exit_flow
    
    else:
        # UNDIRECTED 
        # The random walk is automatically ergodic on undirected graphs,
        # so no teleportation. The stationary distribution is
        # simply proportional to node strength (degree for unweighted)
        
        # Get edge weights (or ones for unweighted graphs)
        # Explicit np.ones with float64 dtype so no extra conversions
        weights = np.array(
            g.es["weight"] if g.is_weighted() else np.ones(g.ecount(), dtype=np.float64)
        )
        total_weight_x2 = 2 * np.sum(weights)

        # Stationary distribution: p_α = strength(α) / 2W
        # The factor of 2 accounts for each undirected edge being counted twice
        p = np.array(
            g.strength(weights="weight" if g.is_weighted() else None)
        ) / total_weight_x2

        # Aggregate to per-community visit rates
        p_mod = np.zeros(max(communities) + 1)
        np.add.at(p_mod, communities, p)

        # For undirected graphs, exit probability is just the normalised
        # weight of edges crossing community boundaries, no teleportation term
        exit_weights = compute_exit_weights(g, communities)
        q_mod = exit_weights / total_weight_x2

    # L(M) = q_⌒·H(Q) + Σ p_i·H(P_i)
    q_sum = np.sum(q_mod)         # total exit probability across all communities
    p_loop = p_mod + q_mod        # per-community total codeword use
    
    L = safe_xlogx(q_sum) - 2 * np.sum(safe_xlogx(q_mod)) \
        - np.sum(safe_xlogx(p)) + np.sum(safe_xlogx(p_loop))
    
    return L